# Fase 1: carga y limpieza de los matriculados en Enfermería (SIIU)

Este notebook deja constancia, paso a paso, de cómo se pasa del Excel original
(`data/raw/Matriculados.xlsx`) al **panel limpio por centro**: una fila por centro
(universidad + código de centro) y curso, con el nº de matriculados en
*Graduado o Graduada en Enfermería*.

No contiene lógica propia: cada paso llama a una función del paquete `prevision`
(`src/prevision/`). Son las mismas funciones que usa el pipeline ETL
(`python -m prevision.pipeline`), de modo que **notebook y pipeline no pueden
divergir**. Al final se comprueba que dan exactamente el mismo panel.

Límites que conviene tener presentes:
- El curso **2025-26 es provisional** (avance) y puede revisarse.
- `NaN` significa *titulación sin dato ese curso*: no hay ceros ni se rellenan huecos.
- Son solo 11 cursos (2015-16 a 2025-26) por serie.

In [1]:
import tempfile
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

from prevision.cleaning import CleaningLog, filter_nursing_degrees, unify_center_series
from prevision.config import NURSING_DEGREE_PATTERN, RAW_FILE, SHEET_CENTERS, SHEET_DEGREES
from prevision.loading import read_raw_sheet, wide_to_long
from prevision.pipeline import run_pipeline
from prevision.validation import compare_university_totals

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 70)

## 1. Estructura del Excel

El Excel tiene tres filas de título. La **fila 3** lleva el curso (solo sobre la columna
*Total* de cada par) y la **fila 4** es la cabecera real: columnas fijas y, después, pares
*Total / % Mujeres* por curso, del más reciente (2025-2026) al más antiguo (2015-2016).
Usamos dos hojas: **"Matriculados Global por centro"** (base del panel) y
**"Matriculados Grado"** (solo para validar totales por universidad).

In [2]:
pd.read_excel(RAW_FILE, sheet_name=SHEET_CENTERS, header=None, nrows=7, dtype=object).iloc[:, :12]

,0,1,2,3,4,5,6,7,8,9,10,11
0,"Serie de estudiantes matriculados de Grado por rama de enseñanza, ...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2025-2026: Datos provisionales Avance,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2025-2026,NaN,2024-2025,NaN
3,Nivel,CCAA (univ presenciales),Universidad,Código centro,Centro,Código Titulación,Titulación,Rama,Total,% Mujeres,Total,% Mujeres
4,Grado y 1o y 2o ciclo,A Distancia de Madrid,A Distancia de Madrid,28053757,Facultad de Ciencias Económicas y Empresariales,2500122,Graduado o Graduada en Administración y Dirección de Empresas,Ciencias Sociales y Jurídicas,NaN,NaN,NaN,NaN
5,Grado y 1o y 2o ciclo,A Distancia de Madrid,A Distancia de Madrid,28053757,Facultad de Ciencias Económicas y Empresariales,2502118,Graduado o Graduada en Economía,Ciencias Sociales y Jurídicas,1,0,2,0
6,Grado y 1o y 2o ciclo,A Distancia de Madrid,A Distancia de Madrid,28053757,Facultad de Ciencias Económicas y Empresariales,2502603,Graduado o Graduada en Empresas y Actividades Turísticas por la Un...,Ciencias Sociales y Jurídicas,94,0.797872,80,0.775


## 2. Lectura en formato ancho

`read_raw_sheet` lee la hoja **sin modificar el fichero**, nombra las columnas en inglés,
conserva los códigos como texto (con sus ceros a la izquierda) y deja los huecos como `NaN`.
Genera `enrolled_<año>` y `pct_women_<año>` por cada curso.

In [3]:
centers_wide = read_raw_sheet(RAW_FILE, SHEET_CENTERS)
degrees_wide = read_raw_sheet(RAW_FILE, SHEET_DEGREES)
print("Hoja de centros:", centers_wide.shape, "| hoja de Grado:", degrees_wide.shape)
centers_wide[["level", "university", "center_code", "center_name", "degree_name", "enrolled_2025", "enrolled_2015"]].head()

Hoja de centros: (17774, 30) | hoja de Grado: (6558, 28)


,level,university,center_code,center_name,degree_name,enrolled_2025,enrolled_2015
0,Grado y 1o y 2o ciclo,A Distancia de Madrid,28053757,Facultad de Ciencias Económicas y Empresariales,Graduado o Graduada en Administración y Dirección de Empresas,NaN,67.0
1,Grado y 1o y 2o ciclo,A Distancia de Madrid,28053757,Facultad de Ciencias Económicas y Empresariales,Graduado o Graduada en Economía,1.0,64.0
2,Grado y 1o y 2o ciclo,A Distancia de Madrid,28053757,Facultad de Ciencias Económicas y Empresariales,Graduado o Graduada en Empresas y Actividades Turísticas por la Un...,94.0,42.0
3,Grado y 1o y 2o ciclo,A Distancia de Madrid,28053757,Facultad de Ciencias Económicas y Empresariales,Graduado o Graduada en Administración y Dirección de Empresas por ...,202.0,247.0
4,Grado y 1o y 2o ciclo,A Distancia de Madrid,28053757,Facultad de Ciencias Económicas y Empresariales,Graduado o Graduada en Marketing por la Universidad a Distancia de...,135.0,53.0


## 3. Filtro de titulación

Reglas (cada una queda en el registro con el nº de filas del Excel antes y después):

- **R1**: nivel de *Grado* (la hoja mezcla también Máster y Doctorado).
- **R2**: solo *Graduado o Graduada en Enfermería*. Se excluyen *Diplomado en Enfermería*,
  los programas conjuntos (PCEO / dobles grados), másteres y doctorados.

**Atención:** el título no siempre es el texto literal. Muchos centros aparecen como
*"Graduado o Graduada en Enfermería por la Universidad de X"*. Con el texto exacto se
perderían casi todos, por eso la regla admite ese sufijo.

In [4]:
literal = (centers_wide["degree_name"] == "Graduado o Graduada en Enfermería").sum()
log = CleaningLog()
centers_filtered = filter_nursing_degrees(centers_wide, log)
print(f"Filas con el texto literal exacto: {literal} | con la regla ampliada: {len(centers_filtered)}")
display(Markdown(log.to_markdown()))

Filas con el texto literal exacto: 8 | con la regla ampliada: 158


| Regla | Descripción | Antes | Después | Diferencia |
|---|---|---:|---:|---:|
| R1-nivel | Nivel que empieza por 'Grado' | 17774 | 7914 | -9860 |
| R2-titulacion | Solo 'Graduado o Graduada en Enfermería' (excluye Diplomado, PCEO/dobles grados, másteres y doctorados) | 7914 | 158 | -7756 |


In [5]:
# Qué se excluye: títulos que mencionan Enfermería pero no son el Grado (por tipo de programa)
mentions_nursing = centers_wide["degree_name"].str.contains("Enfermería", na=False)
is_grade = centers_wide["degree_name"].str.match(NURSING_DEGREE_PATTERN, na=False)
excluded = centers_wide[mentions_nursing & ~is_grade]
excluded["degree_name"].str.split().str[0].value_counts().to_frame("filas excluidas")

,filas excluidas
degree_name,
Máster,86
PCEO,16
Diplomado,6
Programa,6


## 4. Paso a formato largo

`wide_to_long` convierte una columna por curso en una fila por (titulación, curso) y
añade `year_start`, `season` e `is_provisional` (solo 2025-26). Los `NaN` se conservan.

In [6]:
centers_long = wide_to_long(centers_filtered)
print(centers_long.shape, "| cursos:", sorted(centers_long["year_start"].unique().tolist()))
print("Celdas sin dato (NaN):", int(centers_long["enrolled"].isna().sum()))
centers_long[["university", "center_code", "degree_code", "season", "enrolled", "pct_women", "is_provisional"]].head()

(1738, 13)

 | cursos: [2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]
Celdas sin dato (NaN): 430


,university,center_code,degree_code,season,enrolled,pct_women,is_provisional
0,Alfonso X el Sabio Mare Nostrum,29016409,1500526,2025-2026,61.0,0.770492,True
1,Almería,04008558,2501129,2025-2026,NaN,NaN,True
2,Almería,04008558,2504090,2025-2026,528.0,0.795455,True
3,CEU Fernando III,41016334,1500369,2025-2026,77.0,0.857143,True
4,Cádiz,11006577,2501335,2025-2026,451.0,0.833703,True


## 5. Unificación de series por centro

Algunas titulaciones cambian de **código** (cambio de plan), y la serie queda partida.
La regla **R3** unifica por *(universidad, código de centro)* y **suma** los códigos de titulación
del centro por curso (si todos son `NaN`, el resultado sigue siendo `NaN`, nunca 0). Se usa el
código de centro, y no el nombre, porque el nombre puede cambiar con el mismo código
(p. ej. Melilla). Casos típicos:

- **Almería**: un código hasta 2020-21 y otro desde 2021-22, sin solape.
- **Europea de Madrid**: el plan antiguo se extingue (216 → 6) mientras el nuevo crece
  (134 → 549); aquí la suma es lo correcto.

In [7]:
panel = unify_center_series(centers_long, log)
for university, center in [("Almería", "04008558"), ("Europea de Madrid", "28051852")]:
    sel = centers_long[(centers_long["university"] == university)
                       & (centers_long["center_code"] == center) & centers_long["enrolled"].notna()]
    print(f"{university}: matriculados por código de titulación")
    display(sel.pivot_table(index="degree_code", columns="year_start", values="enrolled"))
    print("Panel unificado:")
    display(panel[(panel["university"] == university) & (panel["center_code"] == center)]
            .set_index("year_start")["enrolled"].to_frame().T)
display(Markdown(log.to_markdown()))

Almería: matriculados por código de titulación


year_start,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
degree_code,,,,,,,,,,,
2501129,537.0,552.0,565.0,556.0,562.0,571.0,NaN,NaN,NaN,NaN,NaN
2504090,NaN,NaN,NaN,NaN,NaN,NaN,576.0,545.0,532.0,525.0,528.0


Panel unificado:


year_start,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
enrolled,537.0,552.0,565.0,556.0,562.0,571.0,576.0,545.0,532.0,525.0,528.0


Europea de Madrid: matriculados por código de titulación


year_start,2015,2016,2017,2018,2019,2020,2021,2022,2023
degree_code,,,,,,,,,
2500181,239.0,255.0,216.0,170.0,109.0,27.0,6.0,NaN,NaN
2503601,NaN,NaN,134.0,253.0,377.0,487.0,549.0,565.0,575.0


Panel unificado:


year_start,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
enrolled,239.0,255.0,350.0,423.0,486.0,514.0,555.0,565.0,575.0,NaN,NaN


| Regla | Descripción | Antes | Después | Diferencia |
|---|---|---:|---:|---:|
| R1-nivel | Nivel que empieza por 'Grado' | 17774 | 7914 | -9860 |
| R2-titulacion | Solo 'Graduado o Graduada en Enfermería' (excluye Diplomado, PCEO/dobles grados, másteres y doctorados) | 7914 | 158 | -7756 |
| R3-unificacion | Una serie por (universidad, código de centro): se suman los códigos de titulación (49 celdas centro-curso con solape de códigos). Recuento = series | 158 | 144 | -14 |


## 6. Caso especial: San Juan de Dios (Sevilla) y Pontificia Comillas

El *Centro de Enfermería San Juan de Dios* (Univ. Sevilla) pasa a ser la *EUEF San Juan de Dios*
(Univ. Pontificia Comillas). **Decisión: se dejan como series separadas.** Hay datos de ambos
en los mismos cursos, así que sumarlas sería una interpretación no verificada. Se muestran
juntas para poder comparar. Quien use el panel debe tenerlo presente al interpretar la tendencia.

In [8]:
sjd = panel[((panel["university"] == "Sevilla") & (panel["center_code"] == "41015500"))
            | ((panel["university"] == "Pontificia Comillas") & (panel["center_code"] == "28042772"))]
sjd.pivot_table(index=["university", "center_name"], columns="year_start", values="enrolled")

,year_start,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
university,center_name,,,,,,,,,,,
Pontificia Comillas,Escuela Universitaria de Enfermería y Fisioterapia San Juan de Dios,471.0,473.0,456.0,471.0,478.0,494.0,499.0,635.0,745.0,949.0,1195.0
Sevilla,Centro de Enfermería San Juan de Dios,223.0,233.0,240.0,262.0,273.0,290.0,314.0,330.0,159.0,87.0,67.0


## 7. Validación frente a la hoja de Grado

Control de calidad: la suma de matriculados por universidad y curso en el panel de centros debe
coincidir con la de la hoja "Matriculados Grado" (filtrada con las mismas reglas). Si no cuadrase,
sería señal de filas perdidas o dobles.

In [9]:
degrees_long = wide_to_long(filter_nursing_degrees(degrees_wide, CleaningLog()))
differences = compare_university_totals(centers_long, degrees_long)
n_compared = centers_long.groupby(["university", "year_start"]).ngroups
print(f"{len(differences)} diferencias en {n_compared} comparaciones universidad-curso")
differences

0 diferencias en 770 comparaciones universidad-curso


,university,year_start,enrolled_centers,enrolled_degrees


## 8. Resultado: el panel limpio

Una fila por (centro, curso). El curso 2025-26 va marcado como provisional.

In [10]:
n_centers = panel.drop_duplicates(["university", "center_code"]).shape[0]
print(f"{len(panel)} filas = {n_centers} centros x {panel['year_start'].nunique()} cursos")
print("Cursos provisionales:", sorted(panel.loc[panel["is_provisional"], "season"].unique()))
print("Centros sin dato en algún curso:", int(panel.groupby(["university", "center_code"])["enrolled"].apply(lambda s: s.isna().any()).sum()))
panel.groupby("season")["enrolled"].sum().to_frame("matriculados (total nacional)")

1584 filas = 144 centros x 11 cursos
Cursos provisionales: ['2025-2026']
Centros sin dato en algún curso: 49


,matriculados (total nacional)
season,
2015-2016,45715.0
2016-2017,45553.0
2017-2018,45919.0
2018-2019,47004.0
2019-2020,47994.0
2020-2021,49175.0
2021-2022,50432.0
2022-2023,52048.0
2023-2024,53629.0


## 9. Comprobación: el pipeline da el mismo panel

`run_pipeline` (el ETL que se usará cuando haya datos nuevos) debe producir exactamente el
mismo panel que los pasos anteriores. Se ejecuta en una carpeta temporal para no tocar `data/processed/`.

In [11]:
with tempfile.TemporaryDirectory() as tmp:
    panel_pipeline = run_pipeline(RAW_FILE, Path(tmp))
pd.testing.assert_frame_equal(panel, panel_pipeline)
print("OK: el panel del notebook es idéntico al del pipeline.")

OK: el panel del notebook es idéntico al del pipeline.
